# Final Project_ PCE Growth Regime Classification and Economic Feature Discovery

## Overall Project Architecture :

## Workflow- 20 steps: 

|   Step | Stage          | What we do                             | Main purpose                                                               |
| -----: | -------------- | -------------------------------------- | -------------------------------------------------------------------------- |
|  **1** | Target         | PCE target analysis                    | Examine `Real_PCE_Growth_QoQ` distribution                                 |
|  **2** | Target         | Classification target design           | Determine weak vs strong PCE definition                                    |
|  **3** | Target         | Class-balance analysis                 | Count observations in each class                                           |
|  **4** | Data           | Feature/data audit                     | Verify ~74 predictors, missing values, types, leakage                      |
|  **5** | EDA            | Classification-focused EDA             | Distributions, correlations, class relationships                           |
|  **6** | Validation     | Chronological split                    | Preserve future observations as test data                                  |
|  **7** | Preprocessing  | Build ML pipelines                     | Imputation, scaling and transformations without leakage                    |
|  **8** | Baseline       | Dummy + Logistic Regression            | Establish simple performance benchmarks                                    |
|  **9** | Models         | KNN and SVM                            | Test distance- and margin-based classifiers                                |
| **10** | Models         | Decision Tree                          | Classification + threshold discovery                                       |
| **11** | Ensemble       | Random Forest + Bagging                | Reduce variance and improve tree stability                                 |
| **12** | Boosting       | AdaBoost + Gradient Boosting + XGBoost | Evaluate sequential ensemble methods                                       |
| **13** | Tuning         | Hyperparameter optimization            | Tune models using time-aware validation                                    |
| **14** | Evaluation     | Model comparison                       | Accuracy, precision, recall, F1, confusion matrices, AUC where appropriate |
| **15** | Imbalance      | Imbalance/resampling experiment        | Original vs class weighting vs over/under-sampling vs SMOTE                |
| **16** | Features       | Feature selection                      | Reduce ~74 predictors toward ≤15                                           |
| **17** | Validation     | Reduced-feature retraining             | Compare full vs compact models                                             |
| **18** | Interpretation | Auxiliary methods                      | Permutation importance, PDP/ICE, SHAP                                      |
| **19** | Thresholds     | Economic threshold analysis            | Identify stable Decision Tree/PDP thresholds                               |
| **20** | Conclusion     | Final model + report                   | Findings, recommendations, limitations and next steps                      |


## Step 0 - pre- PCE Target Distribution & Classification Target Design

### Objective 1: 
    Continuous PCE Growth --> Categorical PCE Regime

In [2]:
# ============================================================
# STEP 1 — BUILD PCE GROWTH TARGET DATASET
# Project:
# PCE Growth Regime Classification and Economic Feature Discovery
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [3]:
# ------------------------------------------------------------
# 1. Download Real Personal Consumption Expenditures from FRED
# ------------------------------------------------------------

PCE_URL = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=PCEC96"
)

pce = pd.read_csv(PCE_URL)

print("Raw FRED data:")
display(pce.head())

Raw FRED data:


,observation_date,PCEC96
0,2007-01-01,11181.0
1,2007-02-01,11178.2
2,2007-03-01,11190.7
3,2007-04-01,11201.5
4,2007-05-01,11218.0


In [4]:
# ------------------------------------------------------------
# 2. Clean column names
# ------------------------------------------------------------

pce.columns = ["DATE", "Real_PCE"]

pce["DATE"] = pd.to_datetime(pce["DATE"])
pce["Real_PCE"] = pd.to_numeric(pce["Real_PCE"],errors="coerce")

pce = pce.dropna(subset=["Real_PCE"])

print("\nCleaned data:")
display(pce.head())


Cleaned data:


,DATE,Real_PCE
0,2007-01-01,11181.0
1,2007-02-01,11178.2
2,2007-03-01,11190.7
3,2007-04-01,11201.5
4,2007-05-01,11218.0


In [5]:
# ------------------------------------------------------------
# 3. Set date as index
# ------------------------------------------------------------

pce = pce.set_index("DATE").sort_index()

print("\nFrequency / observations:")
print(pce.index.min(), "to", pce.index.max())
print("Number of observations:", len(pce))


Frequency / observations:
2007-01-01 00:00:00 to 2026-06-01 00:00:00
Number of observations: 234


In [6]:
# ------------------------------------------------------------
# 4. Convert to quarterly observations
# ------------------------------------------------------------
# PCEC96 is already quarterly in FRED.
# We normalize the timestamps to quarter-end so that the target
# will merge cleanly with the rest of our quarterly dataset.

pce.index = pce.index.to_period("Q").to_timestamp("Q")

pce = pce[~pce.index.duplicated(keep="last")]

In [7]:
# ------------------------------------------------------------
# 5. Calculate Quarter-over-Quarter Real PCE Growth
# ------------------------------------------------------------

pce["Real_PCE_Growth_QoQ"] = (
    pce["Real_PCE"]
    .pct_change(fill_method=None)
    * 100
)

In [8]:
# ------------------------------------------------------------
# 6. Remove first missing growth observation
# ------------------------------------------------------------

pce_target = (
    pce[["Real_PCE", "Real_PCE_Growth_QoQ"]]
    .dropna()
    .copy()
)

In [9]:
# ------------------------------------------------------------
# 7. Restrict sample period for our project
# ------------------------------------------------------------

pce_target = pce_target.loc["2007-01-01":"2026-12-31"]

In [10]:
# ------------------------------------------------------------
# 8. Inspect target
# ------------------------------------------------------------

TARGET = "Real_PCE_Growth_QoQ"

In [11]:
print("Target dataset shape:", pce_target.shape)

print("\nFirst observations:")
display(pce_target.head(3))

print("\nLast observations:")
display(pce_target.tail(3))

print("\nTarget data type:")
print(pce_target[TARGET].dtype)

print("\nMissing target values:")
print(pce_target[TARGET].isna().sum())

print("\nTarget summary:")
display(pce_target[TARGET].describe())

Target dataset shape: (77, 2)

First observations:


,Real_PCE,Real_PCE_Growth_QoQ
DATE,,
2007-06-30,11218.5,0.248421
2007-09-30,11309.6,0.812052
2007-12-31,11335.1,0.225472



Last observations:


,Real_PCE,Real_PCE_Growth_QoQ
DATE,,
2025-12-31,16680.7,0.399656
2026-03-31,16730.1,0.296151
2026-06-30,16885.2,0.927072



Target data type:
float64

Missing target values:
0

Target summary:


count    77.000000
mean      0.541667
std       1.095797
min      -6.470559
25%       0.286710
50%       0.564123
75%       0.826364
max       4.657546
Name: Real_PCE_Growth_QoQ, dtype: float64

In [12]:
# ============================================================
# SAVE PCE TARGET DATASET
# ============================================================

from pathlib import Path

# Create a project data folder
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

# File name
output_file = output_dir / "pce_growth_target.csv"

# Save the dataset
pce_target.to_csv(
    output_file,
    index=True,
    index_label="DATE"
)

print("Dataset saved successfully!")
print("File location:", output_file.resolve())
print("Dataset shape:", pce_target.shape)

Dataset saved successfully!
File location: /Users/yc/Python/Coursera/IBM Machine Learning /Course 3/Final Project/data/pce_growth_target.csv
Dataset shape: (77, 2)


In [13]:
# Verify Saved File

check_df = pd.read_csv(
    output_file,
    parse_dates=["DATE"]
)

print("Loaded shape:", check_df.shape)

display(check_df.head())
display(check_df.tail())

print("\nColumns:")
print(check_df.columns.tolist())

print("\nMissing values:")
print(check_df.isna().sum())

Loaded shape: (77, 3)


,DATE,Real_PCE,Real_PCE_Growth_QoQ
0,2007-06-30,11218.5,0.248421
1,2007-09-30,11309.6,0.812052
2,2007-12-31,11335.1,0.225472
3,2008-03-31,11322.1,-0.114688
4,2008-06-30,11340.7,0.164280


,DATE,Real_PCE,Real_PCE_Growth_QoQ
72,2025-06-30,16466.3,0.258160
73,2025-09-30,16614.3,0.898805
74,2025-12-31,16680.7,0.399656
75,2026-03-31,16730.1,0.296151
76,2026-06-30,16885.2,0.927072



Columns:
['DATE', 'Real_PCE', 'Real_PCE_Growth_QoQ']

Missing values:
DATE                   0
Real_PCE               0
Real_PCE_Growth_QoQ    0
dtype: int64
